In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import datetime

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

np.random.seed(42)
keras.utils.set_random_seed(42)

print("Libraries imported successfully.")

In [ ]:
ticker = "AAPL"
start_date = "2020-01-01"
end_date = datetime.datetime.today().strftime("%Y-%m-%d")
sequence_length = 60

print("Ticker:", ticker)
print("Date range:", start_date, "to", end_date)
print("Sequence length:", sequence_length)

In [ ]:
data = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False)

if isinstance(data.columns, pd.MultiIndex):
    data.columns = [col[0] for col in data.columns]

data = data.dropna().copy()

print("Data shape:", data.shape)
data.head()

In [ ]:
print(data.info())
display(data.describe().T)

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data.index, data["Close"], label="Close Price")
plt.title(f"{ticker} Close Price Over Time")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.show()

In [ ]:
numeric_data = data.select_dtypes(include=["int64", "float64"])

plt.figure(figsize=(10, 6))
sns.heatmap(numeric_data.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
feature_df = data.copy()

feature_df["Return_1d"] = feature_df["Close"].pct_change()
feature_df["MA_7"] = feature_df["Close"].rolling(window=7).mean()
feature_df["MA_21"] = feature_df["Close"].rolling(window=21).mean()
feature_df["Volatility_7"] = feature_df["Return_1d"].rolling(window=7).std()

feature_df["Target_Close_Next_Day"] = feature_df["Close"].shift(-1)

feature_df = feature_df.dropna().copy()

selected_features = ["Close", "Volume", "Return_1d", "MA_7", "MA_21", "Volatility_7"]
target_column = "Target_Close_Next_Day"

print("Final engineered dataset shape:", feature_df.shape)
feature_df[selected_features + [target_column]].head()

In [ ]:
n = len(feature_df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = feature_df.iloc[:train_end].copy()
val_df = feature_df.iloc[train_end:val_end].copy()
test_df = feature_df.iloc[val_end:].copy()

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

print("\nTrain date range:", train_df.index.min().date(), "to", train_df.index.max().date())
print("Validation date range:", val_df.index.min().date(), "to", val_df.index.max().date())
print("Test date range:", test_df.index.min().date(), "to", test_df.index.max().date())

In [ ]:
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

X_train_raw = train_df[selected_features].values
y_train_raw = train_df[[target_column]].values

X_val_raw = val_df[selected_features].values
y_val_raw = val_df[[target_column]].values

X_test_raw = test_df[selected_features].values
y_test_raw = test_df[[target_column]].values

X_train_scaled = feature_scaler.fit_transform(X_train_raw)
X_val_scaled = feature_scaler.transform(X_val_raw)
X_test_scaled = feature_scaler.transform(X_test_raw)

y_train_scaled = target_scaler.fit_transform(y_train_raw)
y_val_scaled = target_scaler.transform(y_val_raw)
y_test_scaled = target_scaler.transform(y_test_raw)

print("Scaling completed without leakage.")

In [ ]:
def create_sequences(features_array, target_array, seq_length):
    X_seq = []
    y_seq = []

    for i in range(seq_length, len(features_array)):
        X_seq.append(features_array[i-seq_length:i])
        y_seq.append(target_array[i])

    return np.array(X_seq), np.array(y_seq)

In [ ]:
X_train, y_train = create_sequences(X_train_scaled, y_train_scaled, sequence_length)

X_val_input = np.vstack([X_train_scaled[-sequence_length:], X_val_scaled])
y_val_input = np.vstack([y_train_scaled[-sequence_length:], y_val_scaled])
X_val, y_val = create_sequences(X_val_input, y_val_input, sequence_length)

X_test_input = np.vstack([X_val_scaled[-sequence_length:], X_test_scaled])
y_test_input = np.vstack([y_val_scaled[-sequence_length:], y_test_scaled])
X_test, y_test = create_sequences(X_test_input, y_test_input, sequence_length)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1], X_train.shape[2])),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.2),

    layers.LSTM(32, return_sequences=False),
    layers.Dropout(0.2),

    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),

    layers.Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model.summary()

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_lstm_stock_model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping, checkpoint],
    verbose=1
)

In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(10, 5))
plt.plot(history_df["loss"], label="Train Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
train_pred_scaled = model.predict(X_train, verbose=0)
val_pred_scaled = model.predict(X_val, verbose=0)
test_pred_scaled = model.predict(X_test, verbose=0)

train_pred = target_scaler.inverse_transform(train_pred_scaled)
val_pred = target_scaler.inverse_transform(val_pred_scaled)
test_pred = target_scaler.inverse_transform(test_pred_scaled)

y_train_actual = target_scaler.inverse_transform(y_train)
y_val_actual = target_scaler.inverse_transform(y_val)
y_test_actual = target_scaler.inverse_transform(y_test)

print("Predictions generated successfully.")

In [ ]:
def directional_accuracy(actual, predicted):
    actual = np.array(actual).reshape(-1)
    predicted = np.array(predicted).reshape(-1)

    actual_direction = np.sign(np.diff(actual))
    predicted_direction = np.sign(np.diff(predicted))

    return (actual_direction == predicted_direction).mean()

def evaluate_regression(actual, predicted, label="Model"):
    actual = np.array(actual).reshape(-1)
    predicted = np.array(predicted).reshape(-1)

    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    mape = mean_absolute_percentage_error(actual, predicted) * 100
    dir_acc = directional_accuracy(actual, predicted) * 100

    return pd.DataFrame({
        "Set": [label],
        "RMSE": [rmse],
        "MAE": [mae],
        "MAPE (%)": [mape],
        "Directional Accuracy (%)": [dir_acc]
    })

In [ ]:
train_metrics = evaluate_regression(y_train_actual, train_pred, label="Train - LSTM")
val_metrics = evaluate_regression(y_val_actual, val_pred, label="Validation - LSTM")
test_metrics = evaluate_regression(y_test_actual, test_pred, label="Test - LSTM")

all_metrics = pd.concat([train_metrics, val_metrics, test_metrics], ignore_index=True)
display(all_metrics.round(4))

In [ ]:
test_target_dates = test_df.index[sequence_length:]
naive_baseline = test_df["Close"].iloc[sequence_length-1:-1].values.reshape(-1, 1)

min_len = min(len(naive_baseline), len(y_test_actual), len(test_pred))
naive_baseline = naive_baseline[:min_len]
y_test_actual_aligned = y_test_actual[:min_len]
test_pred_aligned = test_pred[:min_len]
test_target_dates = test_target_dates[:min_len]

baseline_metrics = evaluate_regression(y_test_actual_aligned, naive_baseline, label="Test - Naive Baseline")
lstm_test_metrics = evaluate_regression(y_test_actual_aligned, test_pred_aligned, label="Test - LSTM")

comparison_table = pd.concat([baseline_metrics, lstm_test_metrics], ignore_index=True)
display(comparison_table.round(4))

In [ ]:
results_df = pd.DataFrame({
    "Date": test_target_dates,
    "Actual": y_test_actual_aligned.reshape(-1),
    "Naive_Baseline": naive_baseline.reshape(-1),
    "LSTM_Prediction": test_pred_aligned.reshape(-1)
}).set_index("Date")

plt.figure(figsize=(14, 6))
plt.plot(results_df.index, results_df["Actual"], label="Actual")
plt.plot(results_df.index, results_df["Naive_Baseline"], label="Naive Baseline")
plt.plot(results_df.index, results_df["LSTM_Prediction"], label="LSTM Prediction")
plt.title(f"{ticker} Test Set: Actual vs Baseline vs LSTM")
plt.xlabel("Date")
plt.ylabel("Next Day Closing Price")
plt.legend()
plt.show()

In [ ]:
results_df["Absolute_Error_LSTM"] = np.abs(results_df["Actual"] - results_df["LSTM_Prediction"])
results_df["Absolute_Error_Naive"] = np.abs(results_df["Actual"] - results_df["Naive_Baseline"])

largest_errors = results_df.sort_values("Absolute_Error_LSTM", ascending=False).head(10)
display(largest_errors.round(4))